In [1]:
%cd ../../..

/home/bhkuser/bhklab/katy/ispy2-r2r


In [2]:
from damply import dirs
import pandas as pd
from pathlib import Path

In [3]:
clinical = pd.read_excel(dirs.RAWDATA / "TCIA_ISPY2/clinical/ISPY2-Imaging-Cohort-1-Clinical-Data.xlsx")
clinical = clinical.sort_values(by='Patient_ID', ignore_index=True)
clinical['Patient_ID'] = clinical['Patient_ID'].astype('str')

feats_ISPY = pd.read_csv(dirs.PROCDATA / "TCIA_ISPY2/features/pyradiomics/TCIA_ISPY2_feature_extracted.csv")
feats_ACRIN = pd.read_csv(dirs.PROCDATA / "TCIA_ACRIN-6698/features/pyradiomics/TCIA_ACRIN-6698_feature_extracted.csv")

In [4]:
radiomics = pd.concat([feats_ISPY, feats_ACRIN])

radiomics = radiomics.rename(columns={'patientid': 'Sample_ID'})

radiomics.sort_values(by='Sample_ID', ascending=False)

# Split up the SampleID from R2R into the PatientID and SampleNumber
split_ids = radiomics['Sample_ID'].str.split('_', expand = True)

# Split the PatientID into the dataset name and numeric pat ID
# Use rsplit to handle ACRIN-6698 dataset name
split_patids = split_ids[0].str.rsplit('-', n=1, expand=True)
# Insert these columns into the radiomics dataframe
try:
    radiomics.insert(1, column='Dataset', value=split_patids[0], allow_duplicates=False)
except ValueError:
    print('Dataset already in radiomics dataframe.')

try:
    radiomics.insert(2, column='Patient_ID', value=split_patids[1], allow_duplicates=False)
except ValueError:
    print('PatientID already in radiomics dataframe.')

try:
    radiomics.insert(3, column='Sample_Number', value=split_ids[1], allow_duplicates=False)
except ValueError:
    print('SampleNumber already in radiomics dataframe.')


In [5]:
radiomics_t0 = radiomics.groupby(by='Patient_ID').head(1).reset_index(drop=True)

radiomics_t1 = radiomics.groupby(by='Patient_ID').nth(2).reset_index(drop=True)

# Data Splitting

In [11]:
from sklearn.model_selection import train_test_split

train_pats, test_pats = train_test_split(clinical.Patient_ID,
                            test_size=0.2, 
                            random_state=10, 
                            shuffle=True)

train_pats = train_pats.reset_index(drop=True)
test_pats = test_pats.reset_index(drop=True)

In [18]:
tr_clinical = clinical[clinical['Patient_ID'].isin(train_pats.values)]
tr_radiomics_t0 = radiomics_t0[radiomics_t0['Patient_ID'].isin(train_pats.values)]
tr_radiomics_t1 = radiomics_t1[radiomics_t1['Patient_ID'].isin(train_pats.values)]

test_clinical = clinical[clinical['Patient_ID'].isin(test_pats.values)]
test_radiomics_t0 = radiomics_t0[radiomics_t0['Patient_ID'].isin(test_pats.values)]
test_radiomics_t1 = radiomics_t1[radiomics_t1['Patient_ID'].isin(test_pats.values)]

# AIM 1: Virtual biopsy for key pathologic features and moelecular assay scores

Outcomes: HR + HER2

In [ ]:
aim1_clinical = clinical.loc[['Patient_ID', 'HR']]